In [3]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

In [4]:
#Sampling_FE_data_to_test
df_sampling_train =pd.read_csv('data/train_fe_cat.csv')
df_sampling_test=pd.read_csv('data/test_fe_cat.csv')


In [5]:
df_sampling_train.head()

,Unnamed: 0,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,...,LapNumber_cat,TyreLife_cat,LapTime (s)_cat,LapTime_Delta_cat,Cumulative_Degradation_cat,RaceProgress_cat,Year_cat,PitStop_cat,Stint_cat,Position_cat
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,1.938314,8,...,50,39.0,78.491,-7.564,21.019,0.714286,2022,0,2,8
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,-0.643188,4,...,27,7.0,75.095,-32.617,-223.207,0.346154,2025,1,2,4
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,0.893363,13,...,59,22.0,70.945,-7.540,-100.529,0.819444,2022,0,3,13
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,-1.663862,7,...,2,2.0,94.361,-7.324,-7.324,0.076923,2023,0,1,7
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,-0.802212,2,...,26,6.0,107.878,8.965,-14.139,0.361111,2022,1,3,2


In [7]:
len(df_sampling_train)

439140

In [8]:
config_json = """
{
  "target_column": "PitNextLap",
  "global_random_state": 42,
  "files": [
    {
      "file_id": 1,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.20
    },
    {
      "file_id": 2,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.15
    },
    {
      "file_id": 3,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.25
    },
    {
      "file_id": 4,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.30
    },
    {
      "file_id": 5,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.10
    }
  ]
}
"""


config_json = """
{
  "target_column": "PitNextLap",
  "global_random_state": 42,
  "files": [
    {
      "file_id": 1,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.20
    },
    {
      "file_id": 2,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.15
    },
    {
      "file_id": 3,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.25
    },
    {
      "file_id": 4,
      "chunk_size": 100000,
      "sample_size": 10000,
      "ratio_1": 0.30
    }
  ]
}
"""


In [9]:
import json
import numpy as np
import pandas as pd

# ==========================================================
# 1. JSON CONFIGURATION (Edit distributions per file here)
# ==========================================================


# ==========================================================
# 2. SAMPLING FUNCTION
# ==========================================================
def sample_with_custom_ratio(
    df, target_col, sample_size, ratio_1, random_state=42
):
    """Extracts a stratified sample based on the specific file's ratio_1 config."""
    n_1 = int(sample_size * ratio_1)
    n_0 = sample_size - n_1

    df_1 = df[df[target_col] == 1]
    df_0 = df[df[target_col] == 0]

    if len(df_1) < n_1 or len(df_0) < n_0:
        raise ValueError(
            f"Insufficient data in chunk to fulfill target ratios. "
            f"Required: {n_1} ones (found {len(df_1)}), {n_0} zeros (found {len(df_0)})."
        )

    sample_1 = df_1.sample(n=n_1, random_state=random_state)
    sample_0 = df_0.sample(n=n_0, random_state=random_state)

    sampled_df = (
        pd.concat([sample_1, sample_0])
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    return sampled_df


# ==========================================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================================
# Parse JSON config (or load from file: config = json.load(open('config.json')))
config = json.loads(config_json)
target_col = config["target_column"]
seed = config["global_random_state"]

# Load your 500k dataset (Replace with pd.read_csv("your_dataset.csv"))

df= df_sampling_train.copy()
# Shuffle dataset once globally
df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

# Iterate through each file configuration in the JSON
start_idx = 0
for file_cfg in config["files"]:
    file_id = file_cfg["file_id"]
    chunk_size = file_cfg["chunk_size"]
    sample_size = file_cfg["sample_size"]
    ratio_1 = file_cfg["ratio_1"]

    end_idx = start_idx + chunk_size

    # 1. Extract chunk
    chunk_df = df.iloc[start_idx:end_idx].copy().reset_index(drop=True)

    # 2. Draw target sample using file-specific ratio
    sampled_df = sample_with_custom_ratio(
        df=chunk_df,
        target_col=target_col,
        sample_size=sample_size,
        ratio_1=ratio_1,
        random_state=seed + file_id,
    )

    
   

    # Output verification
    counts = sampled_df[target_col].value_counts()
    percentages = sampled_df[target_col].value_counts(normalize=True) * 100

    sampled_df.rename(columns={target_col: 'ground_truth'}, inplace=True)
    sampled_df['ground_truth'] = sampled_df['ground_truth'].astype(int)
     # 3. Save outputs (Uncomment to write CSV files)
    # chunk_df.to_csv(f"dataset_part_{file_id}_{chunk_size}k.csv", index=False)
    sampled_df.to_csv(f"Sampling_FE_data_to_test/sample_part_{file_id}_{sample_size}k.csv", index=False)

    print(f"=== File {file_id} Configuration ===")
    print(
        f"Target Ratio Config : {ratio_1 * 100:.0f}% '1's / {(1 - ratio_1) * 100:.0f}% '0's"
    )
    print(f"Sample Records Count:\n{counts.to_string()}")
    print(f"Actual Ratio (%):\n{percentages.to_string()}\n")

    # Shift pointer for next dataset chunk
    start_idx = end_idx

=== File 1 Configuration ===
Target Ratio Config : 20% '1's / 80% '0's
Sample Records Count:
PitNextLap
0.0    8000
1.0    2000
Actual Ratio (%):
PitNextLap
0.0    80.0
1.0    20.0

=== File 2 Configuration ===
Target Ratio Config : 15% '1's / 85% '0's
Sample Records Count:
PitNextLap
0.0    8500
1.0    1500
Actual Ratio (%):
PitNextLap
0.0    85.0
1.0    15.0

=== File 3 Configuration ===
Target Ratio Config : 25% '1's / 75% '0's
Sample Records Count:
PitNextLap
0.0    7500
1.0    2500
Actual Ratio (%):
PitNextLap
0.0    75.0
1.0    25.0

=== File 4 Configuration ===
Target Ratio Config : 30% '1's / 70% '0's
Sample Records Count:
PitNextLap
0.0    7000
1.0    3000
Actual Ratio (%):
PitNextLap
0.0    70.0
1.0    30.0



In [10]:
sampled_df.head()

,Unnamed: 0,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,...,LapNumber_cat,TyreLife_cat,LapTime (s)_cat,LapTime_Delta_cat,Cumulative_Degradation_cat,RaceProgress_cat,Year_cat,PitStop_cat,Stint_cat,Position_cat
0,279247,D001,HARD,Dutch Grand Prix,2025,0,27,2,-0.002492,13,...,27,12.0,90.469,-5.972,-208.689,0.375000,2025,0,2,13
1,242627,D067,MEDIUM,Austrian Grand Prix,2023,0,4,1,-1.173317,6,...,4,4.0,70.898,-0.703,-51.912,0.056338,2023,0,1,6
2,29658,MAG,HARD,Canadian Grand Prix,2022,0,33,1,1.611499,14,...,33,33.0,79.824,-28.820,-25.302,0.458333,2022,0,1,14
3,406763,JON,MEDIUM,Japanese Grand Prix,2024,0,8,2,-0.497011,1,...,8,8.0,97.918,-11.810,-20.913,0.102564,2024,0,2,1
4,106424,D036,MEDIUM,Miami Grand Prix,2024,0,16,1,0.397874,3,...,16,16.0,93.902,-5.725,-31.874,0.205128,2024,0,1,3
